# Event Data EDA for Sequence Modeling

Exploratory analysis of an event-level BigQuery table, preparing for next-item prediction (T4Rec-style).  
Fill in the configuration cell below, then run all cells.

---

### Example Usage

Suppose you have an e-commerce clickstream table `my-project.analytics.click_events` with columns:

| Column | Type | Role |
|--------|------|------|
| `visitor_id` | STRING | Sequence key (groups clicks into per-user sequences) |
| `product_sku` | STRING | Item to predict (next-item target) |
| `click_time` | TIMESTAMP | Orders events within a sequence |
| `price_usd` | FLOAT64 | Numerical feature |
| `page_dwell_sec` | FLOAT64 | Numerical feature |
| `category` | STRING | Categorical feature |
| `device` | STRING | Categorical feature |
| `search_term` | STRING | Text feature |

You would fill in the config cell like this:

```python
PROJECT_ID   = "my-project"
DATASET_ID   = "analytics"
TABLE_NAME   = "click_events"

SEQUENCE_KEY_COL = "visitor_id"
ITEM_COL         = "product_sku"
TIMESTAMP_COL    = "click_time"

NUMERICAL_COLS   = ["price_usd", "page_dwell_sec"]
CATEGORICAL_COLS = ["category", "device"]
TEXT_COLS         = ["search_term"]

ENABLE_SAMPLING  = True          # start with sampling for a quick first pass
SAMPLE_LIMIT     = 1_000_000
ENABLE_DEEP_DIVE = False         # turn on after reviewing basic stats
```

Then run all cells. Once the basic stats look good, set `ENABLE_SAMPLING = False` and `ENABLE_DEEP_DIVE = True` for the full analysis.

In [ ]:
# ============================================================
# Cell 0: Setup & Configuration
# ============================================================

# --- USER CONFIG (fill these in) ---
PROJECT_ID = "your-project"
DATASET_ID = "your-dataset"
TABLE_NAME = "your-table"
FULL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

SEQUENCE_KEY_COL = "user_id"        # column that groups events into sequences
ITEM_COL = "item_id"                # target column for next-item prediction
TIMESTAMP_COL = "event_timestamp"   # column that orders events within a sequence

NUMERICAL_COLS = ["price", "duration"]          # numerical feature columns
CATEGORICAL_COLS = ["category", "device_type"]  # categorical feature columns
TEXT_COLS = ["search_query", "item_name"]        # free-text feature columns

# --- TRUNCATION ---
ENABLE_SAMPLING = False    # set True to cap data volume
SAMPLE_LIMIT = 1_000_000  # max rows when sampling is enabled

# --- DEEP DIVE ---
ENABLE_DEEP_DIVE = True   # enable transition & co-occurrence analysis

# --- TOP-N DISPLAY ---
TOP_N = 20  # how many top items/categories to show in charts

# ============================================================
# Imports & BigQuery client
# ============================================================
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

client = bigquery.Client(project=PROJECT_ID)


def run_sql(sql: str) -> pd.DataFrame:
    """Execute a SQL query on BigQuery and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()


def sampled_table() -> str:
    """Return a FROM clause, optionally wrapped in a sampling subquery."""
    if ENABLE_SAMPLING:
        return f"(SELECT * FROM `{FULL_TABLE}` LIMIT {SAMPLE_LIMIT})"
    return f"`{FULL_TABLE}`"


def col_list(cols: list[str]) -> str:
    """Comma-separated, backtick-quoted column list for SQL."""
    return ", ".join(f"`{c}`" for c in cols)


ALL_FEATURE_COLS = NUMERICAL_COLS + CATEGORICAL_COLS + TEXT_COLS
print(f"Table     : {FULL_TABLE}")
print(f"Seq key   : {SEQUENCE_KEY_COL}")
print(f"Item col  : {ITEM_COL}")
print(f"Timestamp : {TIMESTAMP_COL}")
print(f"Sampling  : {'ON (' + str(SAMPLE_LIMIT) + ' rows)' if ENABLE_SAMPLING else 'OFF'}")
print(f"Deep dive : {'ON' if ENABLE_DEEP_DIVE else 'OFF'}")

## 1. Schema Inspection

In [ ]:
# ============================================================
# Cell 1: Schema Inspection
# ============================================================

schema_sql = f"""
SELECT
    column_name,
    data_type,
    is_nullable,
    column_default,
    description
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name = '{TABLE_NAME}'
ORDER BY ordinal_position
"""
schema_df = run_sql(schema_sql)
print(f"=== Schema for {FULL_TABLE} ===")
print(f"Total columns: {len(schema_df)}")
display(schema_df)

# Quick row count + date range
overview_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(`{TIMESTAMP_COL}`) AS earliest_event,
    MAX(`{TIMESTAMP_COL}`) AS latest_event,
    TIMESTAMP_DIFF(MAX(`{TIMESTAMP_COL}`), MIN(`{TIMESTAMP_COL}`), DAY) AS span_days
FROM {sampled_table()}
"""
overview_df = run_sql(overview_sql)
print("\n=== Table Overview ===")
display(overview_df)

## 2. Basic Statistics

In [ ]:
# ============================================================
# Cell 2: Basic Statistics — counts, distinct values, null rates
# ============================================================

# Build a query that computes null count + distinct count for every column
all_cols = [SEQUENCE_KEY_COL, ITEM_COL, TIMESTAMP_COL] + ALL_FEATURE_COLS
# deduplicate while preserving order
seen = set()
all_cols_unique = []
for c in all_cols:
    if c not in seen:
        seen.add(c)
        all_cols_unique.append(c)

null_exprs = []
distinct_exprs = []
for c in all_cols_unique:
    null_exprs.append(f"COUNTIF(`{c}` IS NULL) AS `null__{c}`")
    distinct_exprs.append(f"COUNT(DISTINCT `{c}`) AS `distinct__{c}`")

basic_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    {',\n    '.join(null_exprs)},
    {',\n    '.join(distinct_exprs)}
FROM {sampled_table()}
"""
basic_raw = run_sql(basic_sql)
total_rows = basic_raw["total_rows"].iloc[0]

# Reshape into a readable table
rows = []
for c in all_cols_unique:
    null_count = basic_raw[f"null__{c}"].iloc[0]
    distinct_count = basic_raw[f"distinct__{c}"].iloc[0]
    rows.append({
        "column": c,
        "null_count": int(null_count),
        "null_rate_%": round(100 * null_count / total_rows, 2) if total_rows else 0,
        "distinct_count": int(distinct_count),
        "distinct_rate_%": round(100 * distinct_count / total_rows, 2) if total_rows else 0,
    })

basic_stats_df = pd.DataFrame(rows)
print(f"=== Basic Statistics (total rows: {total_rows:,}) ===")
display(basic_stats_df)

## 3. Sequence Key Analysis
How events group into sequences — critical for choosing sequence length, filtering, and session splitting.

In [ ]:
# ============================================================
# Cell 3: Sequence Key Analysis
# ============================================================

seq_len_sql = f"""
WITH seq_lengths AS (
    SELECT
        `{SEQUENCE_KEY_COL}`,
        COUNT(*) AS seq_len
    FROM {sampled_table()}
    GROUP BY `{SEQUENCE_KEY_COL}`
)
SELECT
    COUNT(*) AS num_sequences,
    COUNTIF(seq_len = 1) AS single_event_sequences,
    ROUND(100 * COUNTIF(seq_len = 1) / COUNT(*), 2) AS single_event_pct,
    MIN(seq_len) AS min_len,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(5)] AS p5,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(25)] AS p25,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(50)] AS p50_median,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(75)] AS p75,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(90)] AS p90,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(95)] AS p95,
    APPROX_QUANTILES(seq_len, 100)[OFFSET(99)] AS p99,
    MAX(seq_len) AS max_len,
    ROUND(AVG(seq_len), 2) AS mean_len
FROM seq_lengths
"""
seq_stats = run_sql(seq_len_sql)
print("=== Sequence Length Statistics ===")
display(seq_stats.T.rename(columns={0: "value"}))

# Distribution histogram (bucketed in SQL for large tables)
seq_hist_sql = f"""
WITH seq_lengths AS (
    SELECT COUNT(*) AS seq_len
    FROM {sampled_table()}
    GROUP BY `{SEQUENCE_KEY_COL}`
),
capped AS (
    -- Cap at p99 for cleaner histogram
    SELECT LEAST(seq_len, (
        SELECT APPROX_QUANTILES(seq_len, 100)[OFFSET(99)] FROM seq_lengths
    )) AS seq_len
    FROM seq_lengths
)
SELECT seq_len, COUNT(*) AS num_sequences
FROM capped
GROUP BY seq_len
ORDER BY seq_len
"""
seq_hist = run_sql(seq_hist_sql)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(seq_hist["seq_len"], seq_hist["num_sequences"], width=0.8, color="steelblue")
ax.set_xlabel("Sequence Length (capped at p99)")
ax.set_ylabel("Number of Sequences")
ax.set_title(f"Sequence Length Distribution ({SEQUENCE_KEY_COL})")
plt.tight_layout()
plt.show()

# Top-N most active sequence keys
top_keys_sql = f"""
SELECT
    `{SEQUENCE_KEY_COL}`,
    COUNT(*) AS event_count,
    MIN(`{TIMESTAMP_COL}`) AS first_event,
    MAX(`{TIMESTAMP_COL}`) AS last_event
FROM {sampled_table()}
GROUP BY `{SEQUENCE_KEY_COL}`
ORDER BY event_count DESC
LIMIT {TOP_N}
"""
top_keys = run_sql(top_keys_sql)
print(f"\n=== Top {TOP_N} Most Active Sequence Keys ===")
display(top_keys)

## 4. Item / Target Distribution
Understanding item popularity is essential for vocabulary sizing, cold-start handling, and evaluation strategy.

In [ ]:
# ============================================================
# Cell 4: Item / Target Distribution
# ============================================================

item_stats_sql = f"""
WITH item_counts AS (
    SELECT
        `{ITEM_COL}`,
        COUNT(*) AS freq
    FROM {sampled_table()}
    GROUP BY `{ITEM_COL}`
)
SELECT
    COUNT(*) AS num_unique_items,
    COUNTIF(freq = 1) AS items_appearing_once,
    ROUND(100 * COUNTIF(freq = 1) / COUNT(*), 2) AS once_pct,
    MIN(freq) AS min_freq,
    APPROX_QUANTILES(freq, 100)[OFFSET(50)] AS median_freq,
    ROUND(AVG(freq), 2) AS mean_freq,
    MAX(freq) AS max_freq
FROM item_counts
"""
item_stats = run_sql(item_stats_sql)
print("=== Item Statistics ===")
display(item_stats.T.rename(columns={0: "value"}))

# Top-N items + cumulative coverage
item_freq_sql = f"""
WITH item_counts AS (
    SELECT
        `{ITEM_COL}`,
        COUNT(*) AS freq
    FROM {sampled_table()}
    GROUP BY `{ITEM_COL}`
),
ranked AS (
    SELECT
        `{ITEM_COL}`,
        freq,
        ROW_NUMBER() OVER (ORDER BY freq DESC) AS rank,
        SUM(freq) OVER (ORDER BY freq DESC) AS cumulative_freq,
        SUM(freq) OVER () AS total_freq
    FROM item_counts
)
SELECT
    `{ITEM_COL}`,
    freq,
    rank,
    ROUND(100 * freq / total_freq, 4) AS pct_of_events,
    ROUND(100 * cumulative_freq / total_freq, 4) AS cumulative_pct
FROM ranked
WHERE rank <= 500  -- pull top 500 for coverage curve
ORDER BY rank
"""
item_freq = run_sql(item_freq_sql)

# Top-N bar chart
top_items = item_freq.head(TOP_N)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(
    top_items[ITEM_COL].astype(str).iloc[::-1],
    top_items["freq"].iloc[::-1],
    color="steelblue"
)
axes[0].set_xlabel("Frequency")
axes[0].set_title(f"Top {TOP_N} Items by Frequency")

# Cumulative coverage curve
axes[1].plot(item_freq["rank"], item_freq["cumulative_pct"], color="steelblue")
for threshold in [80, 90, 95]:
    idx = (item_freq["cumulative_pct"] >= threshold).idxmax()
    rank_at = item_freq.loc[idx, "rank"]
    axes[1].axhline(y=threshold, color="gray", linestyle="--", alpha=0.5)
    axes[1].axvline(x=rank_at, color="gray", linestyle="--", alpha=0.5)
    axes[1].annotate(f"{threshold}% @ rank {rank_at}", xy=(rank_at, threshold),
                     fontsize=9, ha="left", va="bottom")
axes[1].set_xlabel("Item Rank")
axes[1].set_ylabel("Cumulative % of Events")
axes[1].set_title("Item Coverage Curve")

plt.tight_layout()
plt.show()

# Long-tail: log-log frequency plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(item_freq["rank"], item_freq["freq"], color="steelblue")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Item Rank (log)")
ax.set_ylabel("Frequency (log)")
ax.set_title("Item Frequency Long-Tail (log-log)")
plt.tight_layout()
plt.show()

## 5. Temporal Analysis
Time patterns inform session splitting, time-based features, and train/test splits.

In [ ]:
# ============================================================
# Cell 5: Temporal Analysis
# ============================================================

# --- 5a. Events over time (daily) ---
daily_sql = f"""
SELECT
    DATE(`{TIMESTAMP_COL}`) AS event_date,
    COUNT(*) AS event_count,
    COUNT(DISTINCT `{SEQUENCE_KEY_COL}`) AS unique_keys
FROM {sampled_table()}
GROUP BY event_date
ORDER BY event_date
"""
daily = run_sql(daily_sql)

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(daily["event_date"], daily["event_count"], label="Events", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(daily["event_date"], daily["unique_keys"], label="Unique Keys", color="coral", alpha=0.7)
ax1.set_xlabel("Date")
ax1.set_ylabel("Event Count", color="steelblue")
ax2.set_ylabel("Unique Keys", color="coral")
ax1.set_title("Daily Event Volume")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

# --- 5b. Hour-of-day and Day-of-week ---
time_pattern_sql = f"""
SELECT
    EXTRACT(HOUR FROM `{TIMESTAMP_COL}`) AS hour_of_day,
    EXTRACT(DAYOFWEEK FROM `{TIMESTAMP_COL}`) AS day_of_week,
    COUNT(*) AS event_count
FROM {sampled_table()}
GROUP BY hour_of_day, day_of_week
ORDER BY day_of_week, hour_of_day
"""
time_patterns = run_sql(time_pattern_sql)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

hourly = time_patterns.groupby("hour_of_day")["event_count"].sum().sort_index()
ax1.bar(hourly.index, hourly.values, color="steelblue")
ax1.set_xlabel("Hour of Day")
ax1.set_ylabel("Event Count")
ax1.set_title("Events by Hour of Day")

dow = time_patterns.groupby("day_of_week")["event_count"].sum().sort_index()
dow_labels = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
ax2.bar(dow.index, dow.values, color="steelblue")
ax2.set_xticks(range(1, 8))
ax2.set_xticklabels(dow_labels)
ax2.set_xlabel("Day of Week")
ax2.set_ylabel("Event Count")
ax2.set_title("Events by Day of Week")

plt.tight_layout()
plt.show()

# --- 5c. Inter-event time gaps within sequences ---
gap_sql = f"""
WITH ordered AS (
    SELECT
        `{SEQUENCE_KEY_COL}`,
        `{TIMESTAMP_COL}`,
        LAG(`{TIMESTAMP_COL}`) OVER (
            PARTITION BY `{SEQUENCE_KEY_COL}` ORDER BY `{TIMESTAMP_COL}`
        ) AS prev_ts
    FROM {sampled_table()}
),
gaps AS (
    SELECT
        TIMESTAMP_DIFF(`{TIMESTAMP_COL}`, prev_ts, SECOND) AS gap_seconds
    FROM ordered
    WHERE prev_ts IS NOT NULL
)
SELECT
    COUNT(*) AS num_gaps,
    ROUND(AVG(gap_seconds), 1) AS mean_gap_sec,
    APPROX_QUANTILES(gap_seconds, 100)[OFFSET(50)] AS median_gap_sec,
    APPROX_QUANTILES(gap_seconds, 100)[OFFSET(90)] AS p90_gap_sec,
    APPROX_QUANTILES(gap_seconds, 100)[OFFSET(95)] AS p95_gap_sec,
    APPROX_QUANTILES(gap_seconds, 100)[OFFSET(99)] AS p99_gap_sec,
    MAX(gap_seconds) AS max_gap_sec
FROM gaps
"""
gap_stats = run_sql(gap_sql)
print("=== Inter-Event Time Gaps (seconds) ===")
display(gap_stats.T.rename(columns={0: "value"}))

# Gap distribution bucketed (for histogram)
gap_hist_sql = f"""
WITH ordered AS (
    SELECT
        `{SEQUENCE_KEY_COL}`,
        `{TIMESTAMP_COL}`,
        LAG(`{TIMESTAMP_COL}`) OVER (
            PARTITION BY `{SEQUENCE_KEY_COL}` ORDER BY `{TIMESTAMP_COL}`
        ) AS prev_ts
    FROM {sampled_table()}
),
gaps AS (
    SELECT TIMESTAMP_DIFF(`{TIMESTAMP_COL}`, prev_ts, SECOND) AS gap_seconds
    FROM ordered
    WHERE prev_ts IS NOT NULL
),
bucketed AS (
    SELECT
        CASE
            WHEN gap_seconds <= 60 THEN '0-1min'
            WHEN gap_seconds <= 300 THEN '1-5min'
            WHEN gap_seconds <= 900 THEN '5-15min'
            WHEN gap_seconds <= 1800 THEN '15-30min'
            WHEN gap_seconds <= 3600 THEN '30-60min'
            WHEN gap_seconds <= 86400 THEN '1-24hr'
            WHEN gap_seconds <= 604800 THEN '1-7day'
            ELSE '7day+'
        END AS bucket,
        CASE
            WHEN gap_seconds <= 60 THEN 1
            WHEN gap_seconds <= 300 THEN 2
            WHEN gap_seconds <= 900 THEN 3
            WHEN gap_seconds <= 1800 THEN 4
            WHEN gap_seconds <= 3600 THEN 5
            WHEN gap_seconds <= 86400 THEN 6
            WHEN gap_seconds <= 604800 THEN 7
            ELSE 8
        END AS bucket_order
    FROM gaps
)
SELECT bucket, COUNT(*) AS count
FROM bucketed
GROUP BY bucket, bucket_order
ORDER BY bucket_order
"""
gap_hist = run_sql(gap_hist_sql)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(gap_hist["bucket"], gap_hist["count"], color="steelblue")
ax.set_xlabel("Time Gap Bucket")
ax.set_ylabel("Count")
ax.set_title("Inter-Event Time Gap Distribution")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# --- 5d. Sequence time span ---
span_sql = f"""
WITH spans AS (
    SELECT
        `{SEQUENCE_KEY_COL}`,
        TIMESTAMP_DIFF(MAX(`{TIMESTAMP_COL}`), MIN(`{TIMESTAMP_COL}`), SECOND) AS span_sec
    FROM {sampled_table()}
    GROUP BY `{SEQUENCE_KEY_COL}`
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS num_multi_event_seqs,
    ROUND(AVG(span_sec) / 3600, 2) AS mean_span_hours,
    ROUND(APPROX_QUANTILES(span_sec, 100)[OFFSET(50)] / 3600.0, 2) AS median_span_hours,
    ROUND(APPROX_QUANTILES(span_sec, 100)[OFFSET(90)] / 3600.0, 2) AS p90_span_hours,
    ROUND(MAX(span_sec) / 3600.0, 2) AS max_span_hours
FROM spans
"""
span_stats = run_sql(span_sql)
print("=== Sequence Time Spans (hours, multi-event sequences only) ===")
display(span_stats.T.rename(columns={0: "value"}))

## 6. Numerical Feature Profiling

In [ ]:
# ============================================================
# Cell 6: Numerical Feature Profiling
# ============================================================

if NUMERICAL_COLS:
    num_exprs = []
    for c in NUMERICAL_COLS:
        num_exprs.extend([
            f"MIN(`{c}`) AS `{c}__min`",
            f"MAX(`{c}`) AS `{c}__max`",
            f"ROUND(AVG(`{c}`), 4) AS `{c}__mean`",
            f"APPROX_QUANTILES(`{c}`, 100)[OFFSET(50)] AS `{c}__median`",
            f"ROUND(STDDEV(`{c}`), 4) AS `{c}__stddev`",
            f"COUNTIF(`{c}` IS NULL) AS `{c}__nulls`",
            f"COUNT(DISTINCT `{c}`) AS `{c}__distinct`",
        ])

    num_sql = f"""
    SELECT
        COUNT(*) AS total_rows,
        {',\n        '.join(num_exprs)}
    FROM {sampled_table()}
    """
    num_raw = run_sql(num_sql)
    total = num_raw["total_rows"].iloc[0]

    rows = []
    for c in NUMERICAL_COLS:
        rows.append({
            "column": c,
            "min": num_raw[f"{c}__min"].iloc[0],
            "max": num_raw[f"{c}__max"].iloc[0],
            "mean": num_raw[f"{c}__mean"].iloc[0],
            "median": num_raw[f"{c}__median"].iloc[0],
            "stddev": num_raw[f"{c}__stddev"].iloc[0],
            "null_count": int(num_raw[f"{c}__nulls"].iloc[0]),
            "null_%": round(100 * num_raw[f"{c}__nulls"].iloc[0] / total, 2),
            "distinct": int(num_raw[f"{c}__distinct"].iloc[0]),
        })
    num_profile = pd.DataFrame(rows)
    print("=== Numerical Feature Profile ===")
    display(num_profile)

    # Histograms via bucketed SQL
    n_cols_plot = min(len(NUMERICAL_COLS), 4)
    n_rows_plot = (len(NUMERICAL_COLS) + 1) // 2
    fig, axes = plt.subplots(n_rows_plot, 2, figsize=(14, 4 * n_rows_plot))
    axes = np.array(axes).flatten()

    for i, c in enumerate(NUMERICAL_COLS):
        hist_sql = f"""
        WITH bounds AS (
            SELECT
                APPROX_QUANTILES(`{c}`, 100)[OFFSET(1)] AS lo,
                APPROX_QUANTILES(`{c}`, 100)[OFFSET(99)] AS hi
            FROM {sampled_table()}
            WHERE `{c}` IS NOT NULL
        ),
        bucketed AS (
            SELECT
                SAFE_CAST(FLOOR((`{c}` - lo) / NULLIF((hi - lo) / 50.0, 0)) AS INT64) AS bucket,
                lo, hi
            FROM {sampled_table()}, bounds
            WHERE `{c}` IS NOT NULL AND `{c}` BETWEEN lo AND hi
        )
        SELECT
            bucket,
            MIN(lo) AS lo, MIN(hi) AS hi,
            COUNT(*) AS cnt
        FROM bucketed
        GROUP BY bucket
        ORDER BY bucket
        """
        hist_data = run_sql(hist_sql)
        if not hist_data.empty:
            lo = hist_data["lo"].iloc[0]
            hi = hist_data["hi"].iloc[0]
            bin_width = (hi - lo) / 50.0 if hi != lo else 1
            axes[i].bar(
                hist_data["bucket"] * bin_width + lo,
                hist_data["cnt"],
                width=bin_width * 0.9,
                color="steelblue"
            )
        axes[i].set_title(c)
        axes[i].set_ylabel("Count")

    for j in range(len(NUMERICAL_COLS), len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns configured — skipping.")

## 7. Categorical Feature Profiling

In [ ]:
# ============================================================
# Cell 7: Categorical Feature Profiling
# ============================================================

if CATEGORICAL_COLS:
    cat_summary_rows = []
    fig, axes = plt.subplots(len(CATEGORICAL_COLS), 1,
                              figsize=(12, 4 * len(CATEGORICAL_COLS)),
                              squeeze=False)

    for idx, c in enumerate(CATEGORICAL_COLS):
        cat_sql = f"""
        WITH counts AS (
            SELECT
                CAST(`{c}` AS STRING) AS val,
                COUNT(*) AS freq
            FROM {sampled_table()}
            GROUP BY val
        )
        SELECT
            val,
            freq,
            ROUND(100 * freq / SUM(freq) OVER(), 2) AS pct
        FROM counts
        ORDER BY freq DESC
        LIMIT 100
        """
        cat_data = run_sql(cat_sql)

        # Summary stats
        cardinality_sql = f"""
        SELECT
            COUNT(DISTINCT `{c}`) AS cardinality,
            COUNTIF(`{c}` IS NULL) AS null_count,
            COUNT(*) AS total
        FROM {sampled_table()}
        """
        card = run_sql(cardinality_sql).iloc[0]
        cat_summary_rows.append({
            "column": c,
            "cardinality": int(card["cardinality"]),
            "null_count": int(card["null_count"]),
            "null_%": round(100 * card["null_count"] / card["total"], 2) if card["total"] else 0,
        })

        # Bar chart (top N)
        top = cat_data.head(TOP_N)
        ax = axes[idx, 0]
        ax.barh(top["val"].astype(str).iloc[::-1], top["freq"].iloc[::-1], color="steelblue")
        ax.set_title(f"{c} — top {TOP_N} values (cardinality: {int(card['cardinality']):,})")
        ax.set_xlabel("Frequency")

    plt.tight_layout()
    plt.show()

    cat_summary = pd.DataFrame(cat_summary_rows)
    print("=== Categorical Feature Summary ===")
    display(cat_summary)
else:
    print("No categorical columns configured — skipping.")

## 8. Text Feature Profiling

In [ ]:
# ============================================================
# Cell 8: Text Feature Profiling
# ============================================================

if TEXT_COLS:
    text_rows = []
    for c in TEXT_COLS:
        text_sql = f"""
        SELECT
            COUNTIF(`{c}` IS NULL) AS null_count,
            COUNT(*) AS total,
            ROUND(AVG(LENGTH(`{c}`)), 1) AS avg_char_len,
            APPROX_QUANTILES(LENGTH(`{c}`), 100)[OFFSET(50)] AS median_char_len,
            MAX(LENGTH(`{c}`)) AS max_char_len,
            ROUND(AVG(ARRAY_LENGTH(SPLIT(`{c}`, ' '))), 1) AS avg_word_count,
            COUNT(DISTINCT `{c}`) AS distinct_values
        FROM {sampled_table()}
        WHERE `{c}` IS NOT NULL
        """
        stats = run_sql(text_sql).iloc[0]
        text_rows.append({
            "column": c,
            "null_count": int(stats["null_count"]),
            "null_%": round(100 * stats["null_count"] / stats["total"], 2) if stats["total"] else 0,
            "distinct_values": int(stats["distinct_values"]),
            "avg_char_len": stats["avg_char_len"],
            "median_char_len": int(stats["median_char_len"]) if pd.notna(stats["median_char_len"]) else None,
            "max_char_len": int(stats["max_char_len"]) if pd.notna(stats["max_char_len"]) else None,
            "avg_word_count": stats["avg_word_count"],
        })

    text_profile = pd.DataFrame(text_rows)
    print("=== Text Feature Profile ===")
    display(text_profile)

    # Text length histograms
    fig, axes = plt.subplots(len(TEXT_COLS), 1,
                              figsize=(12, 3.5 * len(TEXT_COLS)),
                              squeeze=False)
    for i, c in enumerate(TEXT_COLS):
        len_hist_sql = f"""
        WITH lens AS (
            SELECT LENGTH(`{c}`) AS char_len
            FROM {sampled_table()}
            WHERE `{c}` IS NOT NULL
        ),
        capped AS (
            SELECT LEAST(char_len, (
                SELECT APPROX_QUANTILES(char_len, 100)[OFFSET(95)] FROM lens
            )) AS char_len
            FROM lens
        )
        SELECT char_len, COUNT(*) AS cnt
        FROM capped
        GROUP BY char_len
        ORDER BY char_len
        """
        len_data = run_sql(len_hist_sql)
        axes[i, 0].bar(len_data["char_len"], len_data["cnt"], color="steelblue", width=0.8)
        axes[i, 0].set_title(f"{c} — character length distribution (capped at p95)")
        axes[i, 0].set_xlabel("Character Length")
        axes[i, 0].set_ylabel("Count")

    plt.tight_layout()
    plt.show()

    # Sample values
    for c in TEXT_COLS:
        sample_sql = f"""
        SELECT `{c}`
        FROM {sampled_table()}
        WHERE `{c}` IS NOT NULL
        LIMIT 10
        """
        samples = run_sql(sample_sql)
        print(f"\n--- Sample values: {c} ---")
        for val in samples[c]:
            print(f"  {val}")
else:
    print("No text columns configured — skipping.")

## 9. Data Quality Checks

In [ ]:
# ============================================================
# Cell 9: Data Quality Checks
# ============================================================

# --- 9a. Exact duplicate rows ---
dup_sql = f"""
WITH row_counts AS (
    SELECT *, COUNT(*) OVER (
        PARTITION BY `{SEQUENCE_KEY_COL}`, `{ITEM_COL}`, `{TIMESTAMP_COL}`
    ) AS dup_count
    FROM {sampled_table()}
)
SELECT
    COUNTIF(dup_count > 1) AS duplicate_rows,
    COUNT(*) AS total_rows,
    ROUND(100 * COUNTIF(dup_count > 1) / COUNT(*), 4) AS dup_pct
FROM row_counts
"""
dup_stats = run_sql(dup_sql)
print("=== Duplicate Rows (same key + item + timestamp) ===")
display(dup_stats)

# --- 9b. Duplicate (key, timestamp) pairs — ordering ambiguity ---
ts_dup_sql = f"""
WITH ts_counts AS (
    SELECT
        `{SEQUENCE_KEY_COL}`,
        `{TIMESTAMP_COL}`,
        COUNT(*) AS n
    FROM {sampled_table()}
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS key_ts_pairs_with_ties,
    SUM(n) AS total_events_in_ties,
    MAX(n) AS max_events_per_ts
FROM ts_counts
"""
ts_dup_stats = run_sql(ts_dup_sql)
print("\n=== Timestamp Ordering Ambiguity (same key + timestamp, different events) ===")
display(ts_dup_stats)
if ts_dup_stats["key_ts_pairs_with_ties"].iloc[0] > 0:
    print("WARNING: Some (key, timestamp) pairs have multiple events.")
    print("You may need a tiebreaker column for deterministic ordering.")

# --- 9c. Null key / item rates ---
null_key_sql = f"""
SELECT
    COUNTIF(`{SEQUENCE_KEY_COL}` IS NULL) AS null_key,
    COUNTIF(`{ITEM_COL}` IS NULL) AS null_item,
    COUNTIF(`{TIMESTAMP_COL}` IS NULL) AS null_timestamp,
    COUNT(*) AS total
FROM {sampled_table()}
"""
null_keys = run_sql(null_key_sql)
print("\n=== Null Rates in Core Columns ===")
display(null_keys)
for col_name, col_key in [(SEQUENCE_KEY_COL, "null_key"), (ITEM_COL, "null_item"), (TIMESTAMP_COL, "null_timestamp")]:
    val = null_keys[col_key].iloc[0]
    if val > 0:
        print(f"WARNING: {col_name} has {val:,} NULL values — these rows need handling.")

## 10. [Deep Dive] Item Transitions
Consecutive item pairs reveal navigation/purchase patterns and inform positional embeddings.

In [ ]:
# ============================================================
# Cell 10: [Deep Dive] Item Transitions
# ============================================================

if ENABLE_DEEP_DIVE:
    # --- 10a. Top transitions ---
    transition_sql = f"""
    WITH ordered AS (
        SELECT
            `{SEQUENCE_KEY_COL}`,
            `{ITEM_COL}` AS current_item,
            LEAD(`{ITEM_COL}`) OVER (
                PARTITION BY `{SEQUENCE_KEY_COL}` ORDER BY `{TIMESTAMP_COL}`
            ) AS next_item
        FROM {sampled_table()}
    )
    SELECT
        CAST(current_item AS STRING) AS from_item,
        CAST(next_item AS STRING) AS to_item,
        COUNT(*) AS transition_count
    FROM ordered
    WHERE next_item IS NOT NULL
    GROUP BY from_item, to_item
    ORDER BY transition_count DESC
    LIMIT 50
    """
    transitions = run_sql(transition_sql)
    print(f"=== Top 50 Item Transitions ({ITEM_COL} -> next {ITEM_COL}) ===")
    display(transitions)

    # --- 10b. Self-transition rate ---
    self_trans_sql = f"""
    WITH ordered AS (
        SELECT
            `{ITEM_COL}` AS current_item,
            LEAD(`{ITEM_COL}`) OVER (
                PARTITION BY `{SEQUENCE_KEY_COL}` ORDER BY `{TIMESTAMP_COL}`
            ) AS next_item
        FROM {sampled_table()}
    )
    SELECT
        COUNT(*) AS total_transitions,
        COUNTIF(current_item = next_item) AS self_transitions,
        ROUND(100 * COUNTIF(current_item = next_item) / COUNT(*), 2) AS self_transition_pct
    FROM ordered
    WHERE next_item IS NOT NULL
    """
    self_trans = run_sql(self_trans_sql)
    print("\n=== Self-Transition Rate (same item repeated consecutively) ===")
    display(self_trans)

    # --- 10c. Transition matrix heatmap (top items) ---
    top_items_for_matrix = transitions["from_item"].unique()[:TOP_N]
    items_str = ", ".join(f"'{x}'" for x in top_items_for_matrix)

    matrix_sql = f"""
    WITH ordered AS (
        SELECT
            CAST(`{ITEM_COL}` AS STRING) AS current_item,
            CAST(LEAD(`{ITEM_COL}`) OVER (
                PARTITION BY `{SEQUENCE_KEY_COL}` ORDER BY `{TIMESTAMP_COL}`
            ) AS STRING) AS next_item
        FROM {sampled_table()}
    )
    SELECT current_item, next_item, COUNT(*) AS cnt
    FROM ordered
    WHERE next_item IS NOT NULL
        AND current_item IN ({items_str})
        AND next_item IN ({items_str})
    GROUP BY current_item, next_item
    """
    matrix_data = run_sql(matrix_sql)
    pivot = matrix_data.pivot_table(
        index="current_item", columns="next_item", values="cnt", fill_value=0
    )
    # Normalize rows to probabilities
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(pivot_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
                xticklabels=True, yticklabels=True)
    ax.set_title(f"Transition Probability Matrix (top {len(top_items_for_matrix)} items)")
    ax.set_xlabel("Next Item")
    ax.set_ylabel("Current Item")
    plt.tight_layout()
    plt.show()
else:
    print("Deep dive disabled — set ENABLE_DEEP_DIVE = True to run.")

## 11. [Deep Dive] Repeat & Co-occurrence Patterns

In [ ]:
# ============================================================
# Cell 11: [Deep Dive] Repeat & Co-occurrence Patterns
# ============================================================

if ENABLE_DEEP_DIVE:
    # --- 11a. Repeat items within sequences ---
    repeat_sql = f"""
    WITH seq_items AS (
        SELECT
            `{SEQUENCE_KEY_COL}`,
            COUNT(*) AS total_events,
            COUNT(DISTINCT `{ITEM_COL}`) AS unique_items
        FROM {sampled_table()}
        GROUP BY `{SEQUENCE_KEY_COL}`
    )
    SELECT
        COUNT(*) AS num_sequences,
        COUNTIF(total_events > unique_items) AS seqs_with_repeats,
        ROUND(100 * COUNTIF(total_events > unique_items) / COUNT(*), 2) AS repeat_seq_pct,
        ROUND(AVG(total_events - unique_items), 2) AS avg_repeated_events,
        ROUND(AVG(SAFE_DIVIDE(unique_items, total_events)), 4) AS avg_uniqueness_ratio
    FROM seq_items
    WHERE total_events > 1
    """
    repeat_stats = run_sql(repeat_sql)
    print("=== Repeat Items Within Sequences ===")
    display(repeat_stats.T.rename(columns={0: "value"}))

    # --- 11b. Most-repeated items ---
    most_repeated_sql = f"""
    WITH item_per_seq AS (
        SELECT
            `{SEQUENCE_KEY_COL}`,
            CAST(`{ITEM_COL}` AS STRING) AS item,
            COUNT(*) AS occ
        FROM {sampled_table()}
        GROUP BY 1, 2
        HAVING COUNT(*) > 1
    )
    SELECT
        item,
        COUNT(*) AS num_seqs_with_repeat,
        ROUND(AVG(occ), 2) AS avg_repeats_per_seq,
        MAX(occ) AS max_repeats_in_seq
    FROM item_per_seq
    GROUP BY item
    ORDER BY num_seqs_with_repeat DESC
    LIMIT {TOP_N}
    """
    most_repeated = run_sql(most_repeated_sql)
    print(f"\n=== Top {TOP_N} Most-Repeated Items (within sequences) ===")
    display(most_repeated)

    # --- 11c. Item co-occurrence matrix (top items, within same sequence) ---
    cooc_sql = f"""
    WITH top_items AS (
        SELECT CAST(`{ITEM_COL}` AS STRING) AS item, COUNT(*) AS freq
        FROM {sampled_table()}
        GROUP BY 1
        ORDER BY freq DESC
        LIMIT {TOP_N}
    ),
    seq_items AS (
        SELECT DISTINCT
            `{SEQUENCE_KEY_COL}` AS seq_key,
            CAST(`{ITEM_COL}` AS STRING) AS item
        FROM {sampled_table()}
        WHERE CAST(`{ITEM_COL}` AS STRING) IN (SELECT item FROM top_items)
    )
    SELECT
        a.item AS item_a,
        b.item AS item_b,
        COUNT(DISTINCT a.seq_key) AS co_occurrence_count
    FROM seq_items a
    JOIN seq_items b ON a.seq_key = b.seq_key AND a.item <= b.item
    GROUP BY 1, 2
    """
    cooc = run_sql(cooc_sql)

    # Build symmetric matrix
    cooc_full = pd.concat([
        cooc,
        cooc[cooc["item_a"] != cooc["item_b"]].rename(
            columns={"item_a": "item_b", "item_b": "item_a"}
        )
    ])
    cooc_pivot = cooc_full.pivot_table(
        index="item_a", columns="item_b", values="co_occurrence_count", fill_value=0
    )

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cooc_pivot, cmap="YlOrRd", ax=ax,
                xticklabels=True, yticklabels=True)
    ax.set_title(f"Item Co-occurrence Matrix (top {TOP_N} items, same sequence)")
    plt.tight_layout()
    plt.show()
else:
    print("Deep dive disabled — set ENABLE_DEEP_DIVE = True to run.")

## 12. Summary & Recommendations

Review the outputs above and fill in the findings below to guide sequence construction.

### Key Findings

| Metric | Value | Implication |
|--------|-------|-------------|
| Total rows | ___ | |
| Unique sequence keys | ___ | |
| Unique items | ___ | Vocabulary size |
| Median sequence length | ___ | Informs max_seq_len |
| P95 sequence length | ___ | Upper bound for truncation |
| Single-event sequences % | ___ | May filter these out |
| Median inter-event gap | ___ | Session splitting threshold |
| P95 inter-event gap | ___ | Alternative splitting threshold |
| Self-transition rate | ___ | Consider deduplication |
| Items appearing once | ___ | Cold-start / min-frequency filter |
| Items covering 80% events | ___ | Core vocabulary |

### Recommended Sequence Construction Parameters

- **Min sequence length**: ___ (filter out shorter)
- **Max sequence length**: ___ (truncate longer, take last N)
- **Session split threshold**: ___ seconds (based on gap analysis)
- **Item min frequency**: ___ (filter rare items or map to `<UNK>`)
- **Timestamp features to add**: hour_of_day, day_of_week, time_since_last_event
- **Dedup consecutive same-item events**: yes / no (based on self-transition rate)
- **Null handling**: ___ (drop rows / impute / keep as category)